# Mission 06 · Autonomous warp state machine and deliberation loop

**Squadron Colby · Conversational AI · Winter 2026/27**

Run all cells from a fresh Python 3.11+ kernel. Dependencies are defined before the integrated controller. The four required assertion suites close the notebook.

## Part 2 · Tool contracts and dispatch

Three local tools accept JSON requests. A shared validator enforces the keywords used in their schemas. The dispatcher returns a text observation for both rejection and execution failure.

In [1]:
import csv
import hashlib
import json
import math
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from enum import Enum, auto
from io import StringIO


def navigate_warp_vector(x: float, y: float, z: float) -> str:
    if not any((x, y, z)):
        raise ValueError("A zero vector has no direction")
    return f"Vector accepted: ({x}, {y}, {z})"


def adjust_plasma_shields(intensity: int) -> str:
    if not 0 <= intensity <= 100:
        raise ValueError("Shield intensity outside 0 to 100")
    return f"Shields at {intensity}%"


def query_sensor_array(target: str) -> str:
    readings = {"escape_corridor": "Corridor clear", "homunculus_core": "Plasma flare",
                "eta_carinae_b": "Elevated emission"}
    return readings.get(target.lower(), "Target unresolved")


def contract(**properties):
    return {"type": "object", "properties": properties, "required": list(properties),
            "additionalProperties": False}


NUMBER = {"type": "number"}
TOOLS = {
    "navigate_warp_vector": (navigate_warp_vector, contract(x=NUMBER, y=NUMBER, z=NUMBER)),
    "adjust_plasma_shields": (adjust_plasma_shields, contract(
        intensity={"type": "integer", "minimum": 0, "maximum": 100})),
    "query_sensor_array": (query_sensor_array, contract(target={"type": "string", "minLength": 1})),
}
ENVELOPE = contract(tool={"type": "string"}, arguments={"type": "object"})


def validate(value, schema, path="request"):
    """Validate the JSON Schema keywords used by the four declared contracts."""
    kind = schema["type"]
    if kind == "object":
        if type(value) is not dict:
            raise ValueError(f"{path} must be an object")
        if "properties" in schema:
            missing = set(schema["required"]) - value.keys()
            extra = value.keys() - schema["properties"].keys()
            if missing or (extra and not schema["additionalProperties"]):
                raise ValueError(f"{path}: missing {sorted(missing)}, extra {sorted(extra)}")
            for key, child in schema["properties"].items():
                validate(value[key], child, f"{path}.{key}")
    elif kind == "string":
        if not isinstance(value, str) or len(value) < schema.get("minLength", 0):
            raise ValueError(f"{path} requires a valid string")
    else:
        if type(value) not in (int, float) or not math.isfinite(value):
            raise ValueError(f"{path} requires a finite number")
        if kind == "integer" and int(value) != value:
            raise ValueError(f"{path} requires an integer")
        if not schema.get("minimum", -math.inf) <= value <= schema.get("maximum", math.inf):
            raise ValueError(f"{path} is outside the allowed range")


def unique_object(pairs):
    obj = {}
    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate JSON key {key}")
        obj[key] = value
    return obj


def dispatch(payload):
    try:
        request = json.loads(payload, object_pairs_hook=unique_object)
        validate(request, ENVELOPE)
        name, args = request["tool"], request["arguments"]
        if name not in TOOLS:
            raise ValueError(f"Unknown tool {name}")
        function, schema = TOOLS[name]
        validate(args, schema, "arguments")
        return "Observation: " + function(**args)
    except Exception as exc:
        return f"Tool Execution Error: {exc}"


def request(tool, **arguments):
    return {"tool": tool, "arguments": arguments}

## Part 3 · Trajectory and empirical return

Each action records its starting state, requested action, reward and resulting state. The first reward has weight one. The transition matrix normalizes destination counts separately for each observed sector and action. The twelve jumps below are synthetic: the course folder provides no jump-history file. An unobserved state/action pair has no estimated row.

In [2]:
@dataclass(frozen=True)
class Transition:
    state: str
    action: str
    reward: float
    next_state: str


def discounted_return(trajectory, gamma=0.95):
    if not 0 <= gamma <= 1:
        raise ValueError("Discount must be between 0 and 1")
    return math.fsum(gamma ** t * row.reward for t, row in enumerate(trajectory))


def transition_matrix(records):
    counts = defaultdict(Counter)
    states = set()
    for row in records:
        counts[row.state, row.action][row.next_state] += 1
        states.update((row.state, row.next_state))
    return {key: {s: count[s] / count.total() for s in sorted(states)}
            for key, count in sorted(counts.items())}


# Synthetic jump history, one observation per line. Replace with a CSV file
# opened in text mode to estimate probabilities from another jump history.
HISTORY = """state,action,next_state
outer,warp,clear
outer,warp,shear
outer,warp,clear
outer,warp,clear
outer,hold,outer
outer,hold,shear
shear,warp,shear
shear,warp,clear
shear,warp,shear
shear,hold,outer
shear,hold,outer
shear,hold,shear"""


def read_history(stream):
    return [Transition(row["state"], row["action"], 0, row["next_state"])
            for row in csv.DictReader(stream)]

## Part 4 · Cycle detection

Canonical JSON includes the environment, tool name and arguments. A dictionary tracks repeated fingerprints in expected constant lookup time. Floyd advances one and two links in the observed successor map; matching consecutive periods in the actual trace confirm recurrence. A single repeated key requires three consecutive proposals. Hashing scales with payload size and a Floyd check can scan the bounded history.

In [3]:
def fingerprint(environment, action):
    canonical = json.dumps([environment, action], sort_keys=True, separators=(",", ":"),
                           allow_nan=False)
    return hashlib.sha256(canonical.encode()).hexdigest()


def floyd_period(successor, start):
    """Return the cycle length in a partial deterministic successor map, or zero."""
    slow = fast = start
    while True:
        slow = successor.get(slow)
        fast = successor.get(successor.get(fast))
        if slow is None or fast is None:
            return 0
        if slow == fast:
            break
    period, cursor = 1, successor[slow]
    while cursor != slow:
        period += 1
        cursor = successor[cursor]
    return period


@dataclass
class Sentinel:
    history: list = field(default_factory=list)
    successor: dict = field(default_factory=dict)
    counts: Counter = field(default_factory=Counter)

    def inspect(self, environment, action):
        key = fingerprint(environment, action)
        if self.history:
            self.successor[self.history[-1]] = key
        self.history.append(key)
        self.counts[key] += 1
        if self.counts[key] < 2:
            return None
        period = floyd_period(self.successor, key)
        if period == 1 and self.history[-3:] == [key] * 3:
            return "repeated state/action (period 1)"
        if period > 1 and self.history[-2*period:-period] == self.history[-period:]:
            return f"repeated state/action sequence (period {period})"
        return None

## Part 1 · Controller and deliberation

One call advances one controller transition. The final budget slot is reserved for HALTED. A tool error enters memory before the next deliberation. Shields of at least 60 and a clear corridor permit the simulated warp. Reaching HALTED suppresses subsequent actions.

In [4]:
class AgentState(Enum):
    IDLE = auto()
    PERCEIVING = auto()
    DELIBERATING = auto()
    ACTING = auto()
    OBSERVING = auto()
    REFLECTING = auto()
    HALTED = auto()


S = AgentState
NEXT = {S.IDLE: S.PERCEIVING, S.PERCEIVING: S.DELIBERATING,
        S.DELIBERATING: S.ACTING, S.ACTING: S.OBSERVING,
        S.OBSERVING: S.REFLECTING, S.REFLECTING: S.DELIBERATING}


@dataclass
class Agent:
    max_steps: int = 20
    state: AgentState = S.IDLE
    step_count: int = 0
    shields: int = 0
    corridor_clear: bool = False
    mission_completed: bool = False
    memory: list = field(default_factory=list)
    trajectory: list = field(default_factory=list)
    sentinel: Sentinel = field(default_factory=Sentinel)
    events: list = field(default_factory=list)
    trace: list = field(default_factory=list)
    planned: dict = field(default_factory=dict)
    halt_reason: str = ""

    def __post_init__(self):
        if type(self.max_steps) is not int or not 1 <= self.max_steps <= 20:
            raise ValueError("Budget must be an integer from 1 to 20")

    def environment(self):
        return (self.shields, self.corridor_clear, self.mission_completed)

    def deliberate(self):
        if self.memory and self.memory[-1].startswith("Tool Execution Error:"):
            return request("adjust_plasma_shields", intensity=70)
        if self.shields < 60:
            return request("adjust_plasma_shields", intensity=60)
        if not self.corridor_clear:
            return request("query_sensor_array", target="escape_corridor")
        return request("navigate_warp_vector", x=-0.85, y=0.52, z=0.05)

    def guard(self, environment, action):
        reason = self.sentinel.inspect(environment, action)
        if reason:
            self.halt_reason = reason
            self.events.append("CRITICAL: " + reason)
            self.state = S.HALTED
        return reason

    def act(self):
        before = str(self.environment())
        proposal = self.planned if isinstance(self.planned, dict) else {}
        name, args = proposal.get("tool"), proposal.get("arguments")
        if name == "navigate_warp_vector" and (self.shields < 60 or not self.corridor_clear):
            observation = "Tool Execution Error: Warp preconditions unmet"
        else:
            observation = dispatch(json.dumps(self.planned))
        reward = -5
        if observation.startswith("Observation:"):
            reward = -1
            if name == "adjust_plasma_shields":
                self.shields = args["intensity"]
            elif name == "query_sensor_array" and observation == "Observation: Corridor clear":
                self.corridor_clear = True
            elif name == "navigate_warp_vector":
                self.mission_completed, reward = True, 20
        self.memory.append(observation)
        self.trajectory.append(Transition(before, json.dumps(self.planned, sort_keys=True),
                                          reward, str(self.environment())))

    def step(self, policy=None):
        if self.state == S.HALTED:
            return self.state
        previous = self.state
        self.step_count += 1
        if previous == S.REFLECTING and self.mission_completed:
            self.halt_reason, self.state = "mission complete", S.HALTED
        elif self.step_count >= self.max_steps:
            self.halt_reason, self.state = "step budget", S.HALTED
        else:
            if previous == S.DELIBERATING:
                self.planned = policy(self) if policy else self.deliberate()
                self.guard(self.environment(), self.planned)
            elif previous == S.ACTING:
                self.act()
            if self.state != S.HALTED:
                self.state = NEXT[previous]
        self.trace.append((previous.name, self.state.name))
        return self.state

## Integrated flight and empirical transition matrix

Start with an invalid shield setting. Follow the returned observation through repair, scan and warp. The three values in each recorded environment are shield intensity, corridor clearance and mission completion.

In [5]:
def run_episode(policy=None, max_steps=20):
    agent = Agent(max_steps=max_steps)
    while agent.state != S.HALTED:
        agent.step(policy)
    return agent


def initially_invalid(agent):
    if not agent.memory:
        return request("adjust_plasma_shields", intensity=-1)
    return agent.deliberate()


pilot = run_episode(initially_invalid)
print("Mission:", pilot.halt_reason, "| transitions:", pilot.step_count)
for row, observation in zip(pilot.trajectory, pilot.memory):
    print(row.state, "->", row.next_state, "| reward", row.reward, "|", observation)
print("Discounted episode return:", round(discounted_return(pilot.trajectory), 6))
matrix = transition_matrix(read_history(StringIO(HISTORY)))
for (sector, action), probabilities in matrix.items():
    print(sector, action, {k: round(v, 4) for k, v in probabilities.items()})

Mission: mission complete | transitions: 18
(0, False, False) -> (0, False, False) | reward -5 | Tool Execution Error: arguments.intensity is outside the allowed range
(0, False, False) -> (70, False, False) | reward -1 | Observation: Shields at 70%
(70, False, False) -> (70, True, False) | reward -1 | Observation: Corridor clear
(70, True, False) -> (70, True, True) | reward 20 | Observation: Vector accepted: (-0.85, 0.52, 0.05)
Discounted episode return: 10.295
outer hold {'clear': 0.0, 'outer': 0.5, 'shear': 0.5}
outer warp {'clear': 0.75, 'outer': 0.0, 'shear': 0.25}
shear hold {'clear': 0.0, 'outer': 0.6667, 'shear': 0.3333}
shear warp {'clear': 0.3333, 'outer': 0.0, 'shear': 0.6667}


## Test 1 · Normal completion and budget



In [6]:
assert set(S.__members__) == set("IDLE PERCEIVING DELIBERATING ACTING OBSERVING REFLECTING HALTED".split())
assert run_episode().trace == [
    ("IDLE", "PERCEIVING"), ("PERCEIVING", "DELIBERATING"),
    *([("DELIBERATING", "ACTING"), ("ACTING", "OBSERVING"), ("OBSERVING", "REFLECTING"),
       ("REFLECTING", "DELIBERATING")] * 2),
    ("DELIBERATING", "ACTING"), ("ACTING", "OBSERVING"), ("OBSERVING", "REFLECTING"),
    ("REFLECTING", "HALTED")]
bounded = Agent()
for _ in range(21):
    bounded.step(lambda a: request("query_sensor_array", target=str(a.step_count)))
assert bounded.state == S.HALTED and bounded.step_count == 20 and bounded.halt_reason == "step budget"
assert Agent(max_steps=1).step() == S.HALTED
print("PASS: normal route and 21 calls under the 20-transition budget")

PASS: normal route and 21 calls under the 20-transition budget


## Test 2 · Validation and recovery



In [7]:
for bad in ('{', '[]', '{"tool":"bad","arguments":{}}',
            json.dumps(request("navigate_warp_vector", x=0, y=0, z=0)),
            json.dumps(request("navigate_warp_vector", x=1, y=1)),
            json.dumps(request("adjust_plasma_shields", intensity=True)),
            json.dumps(request("query_sensor_array", target="core", extra=1))):
    assert dispatch(bad).startswith("Tool Execution Error:")
assert pilot.memory[0].startswith("Tool Execution Error:")
assert pilot.memory[1] == "Observation: Shields at 70%" and pilot.mission_completed
broken = run_episode(lambda a: {} if not a.memory else a.deliberate())
assert broken.memory[0].startswith("Tool Execution Error:") and broken.mission_completed
print("PASS: malformed input, validation, execution failure and recovery")

PASS: malformed input, validation, execution failure and recovery


## Test 3 · Discounted return and frequencies



In [8]:
rewards = [10, 0, -5, 20]
sample = [Transition(str(i), "jump", r, str(i+1)) for i, r in enumerate(rewards)]
assert math.isclose(discounted_return(sample), 22.635)
assert all(math.isclose(sum(row.values()), 1) for row in matrix.values())
assert matrix["outer", "warp"]["clear"] == 3/4
assert math.isclose(matrix["shear", "warp"]["clear"], 1/3)
assert all(a.next_state == b.state for a,b in zip(pilot.trajectory, pilot.trajectory[1:]))
print("PASS: G0 = 22.635 and action-conditioned transition frequencies")

PASS: G0 = 22.635 and action-conditioned transition frequencies


## Test 4 · Emergency tripwire

The three-sector fixture counts environment/action samples. In the integrated agent, a decision cycle also includes acting, observing and reflecting.

In [9]:
mock = Agent()
for observation_index, sector in enumerate(["s1","s2","s3"] * 3, start=1):
    if mock.guard(sector, request("query_sensor_array", target="core")):
        break
assert mock.state == S.HALTED and observation_index == 6
assert "period 3" in mock.events[-1] and mock.events[-1].startswith("CRITICAL:")
repeat = run_episode(lambda a: request("adjust_plasma_shields", intensity=-1))
assert repeat.state == S.HALTED and "period 1" in repeat.halt_reason
assert len(repeat.trajectory) == 2
assert floyd_period({"a":"b","b":"c"}, "a") == 0
print("PASS: period 3 at observation 6; identical failed proposal blocked on attempt 3")

PASS: period 3 at observation 6; identical failed proposal blocked on attempt 3
